**cardiac-surgery-and-percutaneous**

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

banner_path = os.environ.get('CARDIAC_SURGERY_BANNER', 'data/Cardiac-Surgery-banner.webp')
if os.path.exists(banner_path):
    img = mpimg.imread(banner_path)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Cardiac-Surgery-banner')
    plt.show()
else:
    print(f'Banner not found: {banner_path}')

In [ ]:
import pandas as pd
import seaborn as sns
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler 
from keras.models import Model
from keras.layers import Input,Dense
from keras.optimizers import Adam


In [ ]:
data=pd.read_csv(os.environ.get("CARDIAC_SURGERY_DATA", "data/cardiac-surgery.csv"))

In [ ]:
data.columns

In [ ]:
data.drop(['Facility ID','Hospital Name','Detailed Region'], axis=1, inplace=True)

In [ ]:
data.head()

In [ ]:
data.isna().sum()

**Fill null values with suitable values using a model**

In [ ]:
from sklearn.impute import KNNImputer
imputer=KNNImputer(n_neighbors=5)
data[['Lower Limit of Confidence Interval','Upper Limit of Confidence Interval']]=imputer.fit_transform(data[['Lower Limit of Confidence Interval','Upper Limit of Confidence Interval']])


In [ ]:
data.isna().sum()

In [ ]:
data.info()

In [ ]:
describe=data.describe()

In [ ]:
describe


In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(describe,annot=True,cmap='coolwarm')
plt.title('data describtion')
plt.show()

In [ ]:
numeric_data=data.select_dtypes(include=['number'])
corr=numeric_data.corr()


**It is a way of visualizing relationships between variables. Colors and numerical values show the strength and direction of these relationships.Colors:**

* Navy blue: Strong positive correlation (if one variable increases, the other tends to increase as well).
* Dark red: Strong negative correlation (if one variable increases, the other tends to decrease).
* White/Beige: Weak or no attachment.
* Numbers: Values of correlation coefficients. 1 means a perfect positive correlation, -1 means an exact negative correlation, 0 means no correlation.
* Conclusion: Look for dark colors and numbers close to 1 or -1 for strong relationships.


**In this picture, for example, there is a strong positive correlation between "observed mortality rate" and "risk-adjusted mortality rate". This means that the higher the observed mortality rate, the risk-adjusted mortality rate tends to increase as well.**

In [ ]:
plt.figure(figsize=(10,4))
sns.heatmap(corr,annot=True,cmap='coolwarm')
plt.title('correlation data')
plt.show()

**Outlier handling**

In [ ]:
numeric_data=data.select_dtypes(include=['number'])
plt.figure(figsize=(8,4))
sns.boxplot(numeric_data,orient='h')


**i will handle the outliers by using Autoencoders**

In [ ]:
from keras.models import Model
from keras.layers import Input, Dense
from sklearn.preprocessing import StandardScaler
import numpy as np


#  i will use StandardScaler for numeric data
scaler = StandardScaler()
Data_selected = scaler.fit_transform(numeric_data)

# determine data dimentions
input_dim = Data_selected.shape[1]
encoding_dim = 32

#  Autoencoder network
input_layer = Input(shape=(input_dim,))
encoded = Dense(encoding_dim, activation='relu')(input_layer)  
decoded = Dense(input_dim, activation='sigmoid')(encoded)  

#  Autoencoder model
autoencoder = Model(input_layer, decoded)

# (Encoder) model
encoder = Model(input_layer, encoded)

# build Decoder model by using encoder input
encoded_input = Input(shape=(encoding_dim,))  
decoder_layer = autoencoder.layers[-1] 
decoder = Model(encoded_input, decoder_layer(encoded_input))  # build Decoder model

# train Autoencoder on data
autoencoder.compile(optimizer='adam', loss='mean_squared_error')
autoencoder.fit(Data_selected, Data_selected, epochs=50, batch_size=256, shuffle=True, validation_split=0.2)


**detailed description of the different types of cases after Procedure**

In [ ]:
Procedure_groupby=data.groupby('Procedure')[['Number of Cases','Number of Deaths']].mean()

In [ ]:
Procedure_groupby

In [ ]:
Procedure_groupby.plot(kind='bar',figsize=(6,5))
plt.title('Average Outcomes By Procedures')
plt.xlabel('procedure')
plt.ylabel('values')
plt.xticks(rotation=90)
plt.legend(title='key')
plt.tight_layout()
plt.show()



In [ ]:
Region_groupby=data.groupby('Region')[['Number of Cases','Number of Deaths']].mean()

In [ ]:
Region_groupby

In [ ]:
Region_groupby.plot(kind='bar')
plt.title('Average Outcomes By Region')
plt.xlabel('Region')
plt.ylabel('values')
plt.tight_layout()
plt.xticks(rotation=90)
plt.legend()
plt.show()

***from this chart, we see that most cases are from  New York State .as a result, it  alse has highest number of deaths.***

In [ ]:
data.columns

In [ ]:
Region_Year=data.groupby('Region')[['Year of Hospital Discharge']].value_counts()

In [ ]:
Region_Year

In [ ]:
plt.figure(figsize=(8,4))
Region_Year.plot(kind='area',stacked=True,alpha=0.5)
plt.xticks(rotation=90)
plt.xlabel('year')
plt.ylabel('cases number')
plt.show()

**In this diagram, we see that NY Metro-NYC, 2010 has the highest number of cases**

In [ ]:
observed_mortality_rate =data.groupby('Procedure')['Observed Mortality Rate'].mean()

In [ ]:
observed_mortality_rate

In [ ]:
plt.figure(figsize=(8,4))
plt.pie(observed_mortality_rate,labels=observed_mortality_rate.index,autopct='%1.1f%%' ,startangle=90 )
plt.title('Observed Mortality Rate by Procedure')
plt.axis('equal')
plt.show()

In [ ]:
mortality_rates  =data.groupby('Procedure')[['Observed Mortality Rate','Observed Mortality Rate']].mean()

In [ ]:
mortality_rates

In [ ]:
mortality_rates.plot(kind='bar',figsize=(8,6),width=0.5)
plt.title('Comparison of Observed vs Expected Mortality Rates by Procedure')
plt.xlabel('Procedure')
plt.ylabel('Mortality Rate')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

**prediction**

In [ ]:
features =['Region', 'Procedure', 'Year of Hospital Discharge', 
     'Number of Cases', 'Number of Deaths', 
     'Expected Mortality Rate']
target =['Comparison Results']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
model=XGBClassifier()
label_enc=LabelEncoder()
data['Region']=label_enc.fit_transform(data['Region'])
data['Procedure'] = label_enc.fit_transform(data['Procedure'])
data['Comparison Results'] = label_enc.fit_transform(data['Comparison Results'])
data['Year of Hospital Discharge'] = pd.to_datetime(data['Year of Hospital Discharge'], errors='coerce').dt.year
X = data[features]
Y = data[target]


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,Y,test_size=0.2 , random_state=42)

In [ ]:
model = XGBClassifier()
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score
y_pred=model.predict(X_test)

In [ ]:
accuracy=accuracy_score(y_pred,y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")